# RIDGE trên Kaggle T4 x2

Run riêng **FP16 v2**, giữ 64 frame ở nguyên độ phân giải; không nối với run Kaggle v1 hoặc 376 kết quả BF16 của server.
Năm biến thể: sigma2, sigma1, multiscale_v1, vote_only, rescue_only.
Mục tiêu 200 câu × 5 = **1.000 kết quả mới**. Dùng lại BLIP score và trọng số Qwen.

1. Chọn **Accelerator → GPU T4 x2**, bật **Internet**.
2. Tải `ridge-kaggle-input.zip` thành một **private Kaggle Dataset**, rồi Add Input vào notebook này.
3. Trong **Add-ons → Secrets**, tạo và cho notebook quyền dùng secret `HF_TOKEN` (read token đã có quyền truy cập LongVideoBench).
4. Chạy lần lượt các cell. `MODE = "pilot"` chạy hai câu / 10 kết quả. Sau khi pilot đạt, đổi thành `"full"` rồi chạy lại cell cuối trong cùng session.

Notebook tải model/thư viện/video tạm vào `/kaggle/temp`; output và checkpoint nhỏ ở `/kaggle/working`.
Đây là bản port mới đã kiểm tra CPU; chưa được xác nhận suy luận trên Kaggle T4 x2. Hai GPU dùng model sharding, không coi VRAM là một GPU cộng gộp tự động.


In [ ]:
from pathlib import Path
import base64, hashlib, json, zipfile, os, sys, subprocess, shutil, urllib.request

WORK = Path('/kaggle/working/ridge-kaggle-v2')
TEMP = Path('/kaggle/temp/ridge-v1')
WORK.mkdir(parents=True, exist_ok=True)
TEMP.mkdir(parents=True, exist_ok=True)
print('Notebook kernel Python:', sys.version.split()[0])
print('The worker will use its own pinned Python 3.12.11 environment.')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

def safe_extract(archive, destination):
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            p = (root / name).resolve()
            if p != root and root not in p.parents:
                raise ValueError('Invalid archive path')
        z.extractall(root)

uploads = list(Path('/kaggle/input').rglob('ridge-kaggle-input.zip'))
BUNDLE = WORK / 'ridge-kaggle-bundle'
if uploads:
    assert len(uploads) == 1, 'Attach exactly one input bundle'
    safe_extract(uploads[0], WORK)
else:
    manifests = list(Path('/kaggle/input').rglob('bundle-manifest.json'))
    assert len(manifests) == 1, 'Attach the private input Dataset (ZIP or expanded files)'
    shutil.copytree(manifests[0].parent, BUNDLE, dirs_exist_ok=True)
inventory = json.loads((BUNDLE / 'bundle-manifest.json').read_text())
vocab_key = 'code/src/evidencelab/_vendor/lens/API_CLIP/clip_prs/utils/vocab/bpe_simple_vocab_16e6.txt.gz'
vocab_url = 'https://raw.githubusercontent.com/openai/CLIP/a1d071733d7111c9c014f024669f959182114e33/clip/bpe_simple_vocab_16e6.txt.gz'
for name, expected in inventory['files'].items():
    target = BUNDLE / name
    if target.is_file():
        continue
    payload = None
    sidecar = inventory.get('transport_base64', {}).get(name)
    if sidecar and (BUNDLE / sidecar).is_file():
        payload = base64.b64decode((BUNDLE / sidecar).read_bytes(), validate=True)
    elif name == vocab_key and expected == '924691ac288e54409236115652ad4aa250f48203de50a9e4722a6ecd48d6804a':
        print('Restoring missing CLIP vocabulary from pinned upstream source...')
        with urllib.request.urlopen(vocab_url, timeout=60) as response:
            payload = response.read(2 * 1024 * 1024)
    if payload is None:
        raise FileNotFoundError(f'Input Dataset is incomplete: {name}. Attach the complete input bundle.')
    assert hashlib.sha256(payload).hexdigest() == expected, f'Restored asset hash mismatch: {name}'
    if target.is_dir():
        raise RuntimeError(f'An expanded archive directory occupies the expected file: {target}')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
    print('Restored exact bundle asset:', name)
for name, expected in inventory['files'].items():
    actual = hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest()
    assert actual == expected, f'Changed bundle file: {name}'
CODE = BUNDLE / 'code'
runner_payload = base64.b64decode('...', validate=True)
assert hashlib.sha256(runner_payload).hexdigest() == '74ab678da14243d22fd4ae44e271830f48816365c8e95c6c6c32eca40c69fc7f'
(CODE / 'scripts/kaggle').mkdir(parents=True, exist_ok=True)
(CODE / 'scripts/kaggle/ridge_cached_ablation.py').write_bytes(runner_payload)
RUN = WORK / 'run'
print('Input verified:', len(inventory['files']), 'files')
print('Kaggle v2 runner installed; existing input Datasets are supported.')


## Môi trường riêng

Cài Python 3.12.11 riêng bằng uv và tạo venv để tránh phụ thuộc phiên bản Python của kernel Kaggle. Torch CUDA 12.6 được ghim riêng. Cell này không nạp mô hình.


In [ ]:
env = os.environ.copy()
env.update(PYTHONPATH=str(CODE / 'src'), PYTHONDONTWRITEBYTECODE='1', PYTHONNOUSERSITE='1',
           PIP_CONFIG_FILE='/dev/null', PIP_NO_CACHE_DIR='1', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='4')
for key in ('PIP_TARGET', 'PIP_PREFIX', 'PIP_USER', 'PYTHONHOME', 'VIRTUAL_ENV'):
    env.pop(key, None)
for key in ('HF_HOME','HF_HUB_CACHE','HUGGINGFACE_HUB_CACHE','HF_MODULES_CACHE','HF_DATASETS_CACHE','HF_XET_CACHE'):
    env[key] = str(TEMP / 'hf' / key.lower())
env.update(HF_HUB_DISABLE_XET='1', HF_HUB_DOWNLOAD_TIMEOUT='120',
           XDG_CACHE_HOME=str(TEMP / 'cache'), TMPDIR=str(TEMP / 'tmp'))
Path(env['TMPDIR']).mkdir(exist_ok=True)

BOOT = TEMP / 'bootstrap-uv-0.8.22'
UV = BOOT / 'bin/uv'
env.update(UV_PYTHON_INSTALL_DIR=str(TEMP / 'python'), UV_CACHE_DIR=str(TEMP / 'uv-cache'))
if not UV.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                    '--target', str(BOOT), 'uv==0.8.22'], env=env, check=True)

VENV = TEMP / 'venv-py312-v2'
PY = VENV / 'bin/python'
if not PY.exists():
    subprocess.run([str(UV), '--no-config', 'venv', '--python', '3.12.11',
                    '--managed-python', '--seed', str(VENV)], env=env, check=True)
subprocess.run([str(PY), '-c',
    "import sys; print('Worker Python:', sys.version); assert sys.version_info[:3] == (3, 12, 11), 'Unexpected worker Python'"],
    env=env, check=True)

def install(args):
    subprocess.run([str(PY), '-m', 'pip', 'install', '--no-cache-dir', *args], env=env, check=True)

marker = VENV / 'ridge-environment-ready-v1'
if not marker.exists():
    install(['pip==25.2','setuptools==80.9.0','wheel==0.45.1'])
    install(['torch==2.8.0','torchvision==0.23.0','--index-url','https://download.pytorch.org/whl/cu126'])
    install(['-r',str(CODE / 'requirements-longvideo-molmo.txt')])
    install(['--no-build-isolation','--no-deps','-e',str(CODE)])
    subprocess.run([str(PY), str(CODE / 'scripts/check_longvideo_dependencies.py')], env=env, check=True)
    marker.write_text('ready')
print('Environment ready')


## Secret và preflight

Không dán token vào code. Đọc `HF_TOKEN` từ Kaggle Secrets. Chỉ kiểm tra GPU, cache và metadata, chưa tải trọng số.


In [ ]:
from kaggle_secrets import UserSecretsClient
env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert env['HF_TOKEN'], 'Missing HF_TOKEN secret'
subprocess.run([str(PY), '-c',
    "import sys; from pathlib import Path; sys.path.insert(0, sys.argv[1]); import ridge_cached_ablation as r; d=r.components.validate_source(Path(sys.argv[2]), Path(sys.argv[3])); print('Preflight passed:', len(d['inventory']), 'cached questions; Kaggle target:', len(d['inventory'])*len(r.VARIANTS), 'new answers')",
    str(CODE / 'scripts/kaggle'), str(BUNDLE / 'source'), str(BUNDLE / 'baseline')], env=env, check=True)
subprocess.run([str(PY), '-c',
    "import sys; sys.path.insert(0, sys.argv[1]); from ridge_cached_ablation import kaggle_profile; print(kaggle_profile())",
    str(CODE / 'scripts/kaggle')], env=env, check=True)


## Tiếp tục một run Kaggle cũ nếu có

Để tiếp tục ở session mới, Add Input một private Dataset chứa `ridge-kaggle-checkpoint.zip` của **v2** đã tải từ session trước. Cell sẽ phục hồi journal Kaggle v2, không nhập journal v1 hoặc server. Luôn gắn lại input bundle gốc.


In [ ]:
archives = list(Path('/kaggle/input').rglob('ridge-kaggle-checkpoint.zip'))
if not (RUN / 'results.sqlite3').exists():
    if archives:
        assert len(archives) == 1, 'Attach only the newest Kaggle checkpoint'
        safe_extract(archives[0], WORK)
    else:
        candidates = []
        for path in Path('/kaggle/input').rglob('contract.json'):
            if (path.parent / 'results.sqlite3').exists():
                data = json.loads(path.read_text())
                assert data.get('schema') == 'kaggle-ridge-fp16-ablation-v2', 'Use a v2 checkpoint; keep v1/server runs separate'
                candidates.append(path.parent)
        assert len(candidates) <= 1, 'Attach only the newest Kaggle checkpoint'
        if candidates:
            shutil.copytree(candidates[0], RUN, dirs_exist_ok=True)
            index = candidates[0].parent / 'metadata/index.json'
            if index.exists():
                (WORK / 'metadata').mkdir(exist_ok=True)
                shutil.copy2(index, WORK / 'metadata/index.json')
    if (RUN / 'contract.json').exists():
        contract = json.loads((RUN / 'contract.json').read_text())
        assert contract['schema'] == 'kaggle-ridge-fp16-ablation-v2', 'Use a v2 checkpoint; keep v1/server runs separate'
        print('Kaggle checkpoint restored')
    else:
        print('Begin a new Kaggle run')
else:
    print('Use current working journal')


## Chạy pilot rồi full

Lần đầu giữ `MODE = "pilot"`. Khi có 10/1.000 kết quả hữu hạn, đủ 64 frame và không lỗi model sharding, đổi sang `MODE = "full"` và chạy lại cell này. Các kết quả pilot nằm trong cùng run, không chạy lại.

Mỗi biến thể được commit vào SQLite; ZIP checkpoint cập nhật sau mỗi câu và lúc worker kết thúc. Bản v2 ghi từng frame RGB nguyên bản ra đĩa, giải phóng decoder rồi mới chạy processor; không giữ danh sách 64 ảnh PIL hoặc tạo np.stack của chúng. Log có các mốc RAM để xác định chỗ tăng bộ nhớ. Có thể Interrupt cell để gửi yêu cầu dừng có kiểm soát. Phiên bị cắt cứng vẫn cần lưu output gần nhất bằng cơ chế Kaggle.


In [ ]:
MODE = 'pilot'  # After pilot succeeds, change to 'full' and rerun this cell.
MAX_SECONDS = 6 * 60 * 60
command = [str(PY), '-u', str(CODE / 'scripts/kaggle/ridge_cached_ablation.py'),
           '--bundle', str(BUNDLE), '--output', str(RUN), '--scratch', str(TEMP),
           '--mode', MODE, '--max-seconds', str(MAX_SECONDS)]
import signal
process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)
log_path = WORK / 'session.log'
try:
    with log_path.open('a') as log:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
    status = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGUSR1)
    print('Stop requested; waiting for the current variant to finish and save…')
    try:
        tail, _ = process.communicate(timeout=180)
        print(tail or '')
    except subprocess.TimeoutExpired:
        print('Worker is still finishing. Do not start another worker in this session.')
        raise
    status = process.returncode

if (RUN / 'summary.json').exists():
    summary = json.loads((RUN / 'summary.json').read_text())
    print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Checkpoint:', WORK / 'ridge-kaggle-checkpoint.zip')
if status not in (0, 75):
    raise RuntimeError(f'Worker exited {status}; retain logs and checkpoint before changing configuration')


## Lưu kết quả

Trong Output tải `ridge-kaggle-v2/ridge-kaggle-checkpoint.zip` và `ridge-kaggle-v2/session.log`, hoặc **Save Version** có lưu outputs. Giữ notebook và input Dataset ở chế độ private. Lần đầu toàn bộ dev200 hoàn thành khi summary có **1.000/1.000**, `complete=true`, mỗi biến thể `n=200`.

Khi hết thời gian, tiếp tục từ ZIP ở session mới với cùng notebook/code và input bundle. Nếu hợp đồng runtime/device map thay đổi, chương trình sẽ dừng thay vì trộn kết quả. Không so accuracy FP16 trên Kaggle với BF16 trên server như một ablation chỉ khác thuật toán.
